# 01 — EMG quality control and anomaly detection

**Phase B** (`docs/roadmap.md`). This notebook establishes the EMG signal-quality
layer that feeds Reborn's safety path, and evaluates the two complementary
detectors on a labelled set of *injected* faults:

1. **Deterministic QC** (`reborn.sensing.emg_qc`) — cheap, always-on, rule-based
   checks the safety layer trusts (dropout, saturation, clipping, amplitude range,
   baseline offset, mains interference). Named failure modes.
2. **Advisory anomaly detection** (`reborn.ml.anomaly`) — a one-class detector
   that flags *"this doesn't look like normal EMG"* without a per-mode rule.
   **Advisory only**: consumed via `reborn.decision.confidence_gate`, never wired
   to actuators, never overriding safety.

> **Data reality.** The public datasets (Ninapro, EMG-EPN-612 — see
> `data/README.md`) are not committed and must be downloaded locally. Until then
> this notebook runs on a **synthetic smoke fixture** that exercises the detector
> plumbing only. Per the roadmap's *real-data-not-synthetic* principle, no
> quantitative claim about real EMG is made here — the `load_emg_windows` cell
> below is the seam where real recordings plug in.

## Setup

In [ ]:
import numpy as np

from reborn.sensing.emg_qc import assess_quality_report
from reborn.sensing.features import extract_features
from reborn.sensing import corruption
from reborn.ml.anomaly import AnomalyDetector

RNG = np.random.default_rng(0)
SAMPLE_RATE = 1000.0   # Hz
WINDOW = 400           # samples per analysis window
MODES = corruption.FAULT_MODES
print("fault modes:", MODES)

## 1. EMG source  —  *the real-data seam*

`load_emg_windows` returns clean (rest / low-activity) single-channel EMG windows.
**To use real data:** replace its body with a Ninapro / EMG-EPN-612 loader and set
`USING_REAL_DATA = True`. Everything downstream is unchanged.

In [ ]:
def load_emg_windows(n_windows=300, window=WINDOW):
    """Clean EMG windows, shape (n_windows, window).

    SEAM: swap the body for a real loader (see data/README.md). The synthetic
    fallback is a smoke fixture — detector plumbing only, not a result.
    """
    return 0.1 * RNG.standard_normal((n_windows, window))


USING_REAL_DATA = False  # flip once load_emg_windows reads a downloaded dataset
clean = load_emg_windows()
print(f"clean windows: {clean.shape}   real_data={USING_REAL_DATA}")
if not USING_REAL_DATA:
    print("NOTE: synthetic smoke fixture in use — plumbing check only.")

## 2. Deterministic QC vs. injected faults

For each fault mode we corrupt clean windows with `reborn.sensing.corruption`
(the inverse of the QC checks) and measure the detection rate. Expect the
*hard, nameable* modes to be caught near-always; `noise_burst` is deliberately
**not** the deterministic layer's job — that's what the anomaly detector is for.

In [ ]:
ref = assess_quality_report(clean[0], sample_rate=SAMPLE_RATE)
print(f"{'clean reference':<18} valid={ref.valid}  failures={ref.failures}\n")

print(f"{'mode':<16}{'detected':<12}{'example failures'}")
print("-" * 52)
for mode in MODES:
    detected = 0
    example = ()
    for w in clean[:100]:
        rep = assess_quality_report(corruption.corrupt(w.copy(), mode), sample_rate=SAMPLE_RATE)
        if not rep.valid:
            detected += 1
            example = rep.failures
    print(f"{mode:<16}{detected}/100{'':<6}{example}")

## 3. Advisory anomaly detector

Fit the one-class detector on clean feature vectors (RMS / MAV / ZCR), then score
held-out clean windows and each corruption. The clean flag rate should sit near
the `contamination` level; corrupted windows — including `noise_burst`, which the
deterministic layer skips — should score higher and flag more often.

In [ ]:
def feature_row(x):
    f = extract_features(x)
    return [f["rms"], f["mav"], f["zcr"]]


split = len(clean) // 2
train = np.array([feature_row(w) for w in clean[:split]])
held_out = clean[split:]

detector = AnomalyDetector(contamination=0.025).fit(train)
clean_rate = np.mean([detector.score(feature_row(w)).is_anomalous for w in held_out])
print(f"Mahalanobis threshold: {detector.threshold:.3f}")
print(f"clean flag rate (held-out): {clean_rate:.1%}\n")

print(f"{'mode':<16}{'flag rate':<12}{'mean score'}")
print("-" * 42)
for mode in MODES:
    s = [detector.score(feature_row(corruption.corrupt(w.copy(), mode))) for w in held_out]
    rate = np.mean([a.is_anomalous for a in s])
    mean_score = float(np.mean([a.score for a in s]))
    print(f"{mode:<16}{rate:<12.1%}{mean_score:.2f}")

## 4. Reading & next steps

**Two layers, on purpose.** The deterministic checks give the safety path cheap,
explainable, named guarantees; the advisory detector adds coverage for
degradations that no single threshold names (e.g. `noise_burst`). The detector's
output stays advisory — it lowers confidence through
`reborn.decision.confidence_gate`, and **low confidence reduces assist, never
increases it** (`docs/safety.md`, `docs/research/research-context.md` §5.3).

**To turn this from plumbing into a result:**
1. Download Ninapro / EMG-EPN-612 (`data/README.md`) and implement the
   `load_emg_windows` seam (use rest windows as the clean reference).
2. Report detection rates and false-positive rates on *real* windows, and
   characterise cross-session drift of the clean-signal statistics (bridge to
   notebook `03_drift_fewshot`).
3. Add plots (amplitude traces per fault, score distributions) — matplotlib is
   an optional dep; this notebook stays text-only so it runs anywhere.